### **Notebook 1 (etapa 5): SHAP aplicado al modelo óptimo de mortalidad (Random Forest)**

In [ ]:
import os  # Interacción con el sistema operativo (creación de directorios y manejo de rutas)
import gc  # Recolección de basura (Garbage Collector) para liberar memoria RAM durante procesos pesados
import time  # Medición de tiempos de ejecución para monitoreo del rendimiento
import pickle  # Serialización nativa de Python (importado por compatibilidad de dependencias)
import joblib  # Carga (deserialización) del modelo de Machine Learning pre-entrenado
import numpy as np  # Facilita la realización de cálculos numéricos avanzados y manejo de matrices (arrays)
import pandas as pd  # Permite el manejo y análisis de estructuras de datos tabulares (DataFrames)
import shap  # Biblioteca principal basada en teoría de juegos para la explicabilidad de modelos de ML
import matplotlib.pyplot as plt  # Biblioteca para la creación y exportación de visualizaciones gráficas
import warnings  # Control de advertencias del sistema
warnings.filterwarnings("ignore", category=UserWarning)  # Suprime advertencias no críticas para limpiar la consola

def generar_explicabilidad_shap_mortalidad_rf():
    """
    Descripción:
        Ejecuta un pipeline unificado de explicabilidad (SHAP) sobre el modelo Random Forest 
        entrenado para predecir MORTALIDAD. Calcula los valores SHAP en bloques para optimizar RAM, 
        genera reportes absolutos y porcentuales, crea gráficos de resumen (Summary Plots) y 
        dependencia, y extrae la direccionalidad del impacto clínico (aumenta o reduce el riesgo).
        Finalmente, compara el comportamiento del modelo en la cohorte global vs la oncológica.

    Entradas:
        - Ninguna explícita: La función consume directamente los datasets de prueba y el modelo .pkl desde el disco duro.

    Salidas:
        - None: La función no retorna variables en memoria, pero guarda en el disco:
            1. Matrices crudas de SHAP (.npy) como respaldo.
            2. Reportes CSV con impacto absoluto, porcentual y análisis direccional.
            3. Gráficos PNG de resumen (Top 20 general y categórico) y paneles de dependencia.
            4. Un reporte CSV cruzado con las diferencias de impacto entre Global vs Oncológico.
    """
    # Definir la variable objetivo y la clase de interés para el análisis direccional
    target_name = 'MORTALIDAD'
    idx_clase_alta = 1  # Clase 1 corresponde a "Fallecido"
    nombre_efecto_str = 'Mortalidad (Clase 1)'

    # -------------------------------------------------------------------------
    # CONFIGURACIÓN DE RUTAS Y PARÁMETROS
    # -------------------------------------------------------------------------
    # Definir el directorio de los datos finales preprocesados
    dir_datos = "../../Datos/Datasets Finales"
    # Definir el directorio donde se guardó el modelo Random Forest
    dir_modelos = "../../Resultados/Resultados (etapa 3 y 4)/Random_Forest"
    # Definir y crear el directorio raíz para los resultados de explicabilidad (Fase 5)
    dir_base_resultados = f"../../Resultados/Resultados (etapa 5)/SHAP_{target_name}"
    os.makedirs(dir_base_resultados, exist_ok=True)
    
    # Construir la ruta exacta hacia el modelo óptimo
    nombre_modelo = f"Modelo_Optimo_RF_{target_name}.pkl"
    ruta_modelo = os.path.join(dir_modelos, nombre_modelo)
    
    # Listas de variables a excluir del análisis y de variables estrictamente numéricas
    cols_excluir = ['CONSUMO_RECURSOS', 'SEVERIDAD', 'MORTALIDAD', 'CATEGORIA_CANCER']
    vars_num = ['CANTIDAD_TRASLADOS', 'CARGA_ONCOLOGICA', 'DIAS_ESTADIA', 'EDAD', 'NUM_COMORBILIDADES', 'NUM_PROCEDIMIENTOS']
    
    # Imprimir encabezado del proceso
    print("="*80)
    print(f"INICIANDO FASE 5: SHAP BINARIO - TARGET: {target_name}")
    print(f"Foco clínico de análisis direccional: {nombre_efecto_str}")
    print(f"Hora de inicio: {time.strftime('%Y-%m-%d %H:%M:%S')}")
    print("="*80)
    
    # Validar la existencia física del modelo antes de continuar
    if not os.path.exists(ruta_modelo):
        print(f"ERROR: No se encontró el modelo en la ruta: {ruta_modelo}")
        return
        
    print(f"-> Cargando modelo óptimo pre-entrenado ({nombre_modelo})...")
    # Deserializar y cargar el modelo en RAM
    modelo_rf = joblib.load(ruta_modelo)
        
    # Extraer el estimador base real si el modelo fue guardado dentro de un GridSearchCV, Pipeline o Calibrador
    if hasattr(modelo_rf, 'best_estimator_'): modelo_rf = modelo_rf.best_estimator_
    if hasattr(modelo_rf, 'steps'): modelo_rf = modelo_rf.steps[-1][1]
    if hasattr(modelo_rf, 'calibrated_classifiers_'): modelo_rf = modelo_rf.calibrated_classifiers_[0].estimator

    # Recuperar los nombres exactos de las variables que el modelo observó al ser entrenado
    if hasattr(modelo_rf, 'feature_names_in_'):
        features = modelo_rf.feature_names_in_.tolist()
    else:
        # Fallback: leer una fila del dataset para inferir las columnas si el modelo no las guardó
        df_dummy = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), nrows=1)
        features = [c for c in df_dummy.columns if c not in cols_excluir]

    print("-> Inicializando SHAP TreeExplainer nativo...")
    # Instanciar el explicador de árboles de SHAP, optimizado para Random Forest
    explainer = shap.TreeExplainer(modelo_rf)
    
    # -------------------------------------------------------------------------
    # FUNCIÓN INTERNA DE PROCESAMIENTO
    # -------------------------------------------------------------------------
    def procesar_enfoque_shap_binario(df_origen, tipo_enfoque, nombre_carpeta_sub):
        # Anunciar inicio del procesamiento para el enfoque actual (Global u Oncológico)
        print(f"\n--- Procesando enfoque: {tipo_enfoque.upper()} (Datos: {len(df_origen)}) ---")
        
        # Crear subdirectorios específicos para los resultados y gráficos de este enfoque
        dir_sub_enfoque = os.path.join(dir_base_resultados, nombre_carpeta_sub)
        dir_dependence = os.path.join(dir_sub_enfoque, "Dependence_Plots")
        os.makedirs(dir_dependence, exist_ok=True)
        
        # Alinear y convertir la matriz predictora a float32 para optimizar el cómputo de SHAP
        X_shap = df_origen[features].astype('float32')
        # Liberar el dataframe de origen de la memoria RAM
        del df_origen; gc.collect()
        
        # Iniciar cronómetro del cálculo SHAP
        inicio_time = time.time()
        
        # Configurar cálculo en bloques (batches) de 500 pacientes para evitar colapso de RAM (Swap Thrashing)
        batch_size = 500
        resultados_list = []
        n_batches = (len(X_shap) // batch_size) + (1 if len(X_shap) % batch_size != 0 else 0)
        
        # Bucle de inferencia SHAP por lotes
        for i in range(0, len(X_shap), batch_size):
            # Extraer el bloque actual de pacientes
            batch = X_shap.iloc[i:i+batch_size]
            # Imprimir progreso cada 10 bloques
            if (i // batch_size + 1) % 10 == 0 or (i // batch_size + 1) == 1:
                print(f"      -> Procesando bloque {i//batch_size + 1} de {n_batches}...")
            
            # Calcular valores SHAP del bloque (approximate=True agiliza sin perder la tendencia general en RF)
            shap_output = explainer.shap_values(batch, check_additivity=False, approximate=True)
            
            # Reestructurar la salida a un tensor 3D estandarizado (Muestras, Variables, Clases)
            if isinstance(shap_output, list):
                shap_mat_batch = np.stack(shap_output, axis=2)
            elif len(shap_output.shape) == 3:
                shap_mat_batch = shap_output
            else:
                # Si SHAP retorna 2D (solo clase positiva), inferir la negativa para estructurar en 3D
                shap_mat_batch = np.stack([shap_output * -1, shap_output], axis=2)
                
            # Almacenar el bloque procesado y limpiar variables temporales
            resultados_list.append(shap_mat_batch)
            del batch, shap_output, shap_mat_batch; gc.collect()
            
        # Concatenar todos los bloques en una única matriz maestra de SHAP
        matriz_shap = np.concatenate(resultados_list, axis=0)
        print(f"   -> SHAP completado en {round((time.time() - inicio_time)/60, 2)} minutos.")
        
        # --- FILTRO ONCOLÓGICO DE CONSTANTES ---
        # Si estamos en la cohorte estrictamente oncológica, eliminar variables con varianza 0 (ej. SIN_CANCER)
        if "onco" in nombre_carpeta_sub.lower():
            varianzas = X_shap.var()
            cols_a_eliminar = varianzas[varianzas == 0].index.tolist()
            # Forzar la eliminación de la categoría SIN_CANCER si por error sigue presente
            if 'CATEGORIA_CANCER_SIN_CANCER' in X_shap.columns and 'CATEGORIA_CANCER_SIN_CANCER' not in cols_a_eliminar:
                cols_a_eliminar.append('CATEGORIA_CANCER_SIN_CANCER')
                
            # Si existen columnas constantes, purgarlas de X_shap y de la matriz SHAP
            if cols_a_eliminar:
                idx_a_eliminar = [X_shap.columns.get_loc(col) for col in cols_a_eliminar]
                X_shap = X_shap.drop(columns=cols_a_eliminar)
                matriz_shap = np.delete(matriz_shap, idx_a_eliminar, axis=1)
                print(f"      FILTRO: Se excluyeron {len(cols_a_eliminar)} variables constantes.")
        
        # Identificar número de clases y definir el sufijo de los archivos de salida
        n_clases = matriz_shap.shape[2]
        sufijo_archivo = "GLOBAL" if "global" in nombre_carpeta_sub.lower() else "ONCO"
        
        # 1. Guardar respaldo (.npy) de la matriz multidimensional cruda
        ruta_npy = os.path.join(dir_sub_enfoque, f"BACKUP_MATRIZ_GLOBAL_{target_name}_{sufijo_archivo}.npy")
        np.save(ruta_npy, matriz_shap)
        
        # 2. Exportar CSV Numérico Absoluto
        # Promediar el impacto absoluto de cada variable a través de todos los pacientes
        shap_abs = np.abs(matriz_shap).mean(axis=0) 
        # Sumar el impacto a través de todas las clases para obtener un ranking unificado
        impacto_total = shap_abs.sum(axis=1)
        # Formatear y exportar la tabla resumen ordenada de mayor a menor impacto
        columnas_csv = [f"Clase_{i}" for i in range(n_clases)]
        df_shap_imp = pd.DataFrame(shap_abs, index=X_shap.columns, columns=columnas_csv)
        df_shap_imp['Impacto_Total'] = impacto_total
        df_shap_imp = df_shap_imp.sort_values(by='Impacto_Total', ascending=False)
        ruta_csv = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}.csv")
        df_shap_imp.to_csv(ruta_csv, index_label='Variable')

        # 3. Exportar CSV Porcentual
        print("   -> Generando matriz de importancias porcentuales...")
        df_shap_porcentajes = df_shap_imp.copy()
        # Convertir los valores absolutos a porcentajes relativos para facilitar la interpretación del cliente
        cols_num_pct = df_shap_porcentajes.select_dtypes(include=['number']).columns
        for col in cols_num_pct:
            suma_total = df_shap_porcentajes[col].sum()
            if suma_total > 0:
                df_shap_porcentajes[col] = (df_shap_porcentajes[col] / suma_total) * 100
        
        # Exportar el ranking porcentual a disco
        ruta_csv_pct = os.path.join(dir_sub_enfoque, f"SHAP_Valores_{target_name}_{sufijo_archivo}_PORCENTAJES.csv")
        df_shap_porcentajes.to_csv(ruta_csv_pct, index_label='Variable')
        
        # Guardar copia formateada para usarla luego en la comparativa cruzada
        df_retorno = df_shap_porcentajes.reset_index().rename(columns={'index': 'Variable'})
        
        # 4. Gráficos Summary Plots
        # Generar gráfico de barras horizontales con el Top 20 de variables predictoras generales
        plt.figure(figsize=(12, 8))
        df_top20 = df_shap_imp.head(20).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='bwr', ax=plt.gca())
        plt.title(f'Top 20 Variables SHAP - Enfoque {tipo_enfoque} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_General_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()
        
        # Generar gráfico similar exclusivo para el Top 20 de variables categóricas
        plt.figure(figsize=(12, 8))
        vars_cat_ohe = [col for col in df_shap_imp.index if col not in vars_num]
        df_top20_cat = df_shap_imp.loc[vars_cat_ohe].head(20).drop(columns=['Impacto_Total']).iloc[::-1]
        df_top20_cat.plot(kind='barh', stacked=True, figsize=(12, 8), cmap='coolwarm', ax=plt.gca())
        plt.title(f'Top 20 Variables Categóricas SHAP - Enfoque {tipo_enfoque} ({target_name})', fontsize=14)
        plt.xlabel('Valor SHAP absoluto promedio')
        plt.tight_layout()
        plt.savefig(os.path.join(dir_sub_enfoque, f"SHAP_Summary_Categoricas_{target_name}_{sufijo_archivo}.png"), dpi=300)
        plt.close()

        # Aislar el tensor SHAP exclusivo de la clase 1 (Fallecidos) para el análisis direccional
        matriz_clase_alta = matriz_shap[:, :, idx_clase_alta]
        
        # 5. Análisis Direccional: Variables Numéricas (Cuartiles)
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str}) para variables numéricas...")
        rangos_direccionales = []
        for v_num in vars_num:
            if v_num in X_shap.columns:
                # Intentar agrupar a los pacientes en cuartiles perfectos; si fallan por duplicados, usar cortes equitativos
                try: bins_serie = pd.qcut(X_shap[v_num], q=4, duplicates='drop')
                except: bins_serie = pd.cut(X_shap[v_num], bins=4)
                
                # Calcular el promedio de impacto SHAP para cada cuartil para ver si aporta o protege
                idx_var = X_shap.columns.get_loc(v_num)
                for rango in bins_serie.cat.categories:
                    indices_rango = (bins_serie == rango)
                    n_pacientes = indices_rango.sum()
                    if n_pacientes > 0:
                        promedio_crudo = matriz_clase_alta[indices_rango, idx_var].mean()
                        if promedio_crudo > 0: efecto = "Aumenta Mortalidad (+)"
                        elif promedio_crudo < 0: efecto = "Protector / Supervivencia (-)"
                        else: efecto = "Neutral"

                        # Guardar el hallazgo clínico del cuartil
                        rangos_direccionales.append({
                            "Variable": v_num, "Rango": str(rango), "N_Pacientes": n_pacientes,
                            f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo, "Efecto_Clinico": efecto
                        })
                        
        # Exportar el reporte direccional numérico
        df_rangos_num = pd.DataFrame(rangos_direccionales)
        df_rangos_num.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Numericas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # 6. Análisis Direccional: Variables Categóricas (OHE 0 vs 1)
        print(f"   -> Extrayendo impacto direccional ({nombre_efecto_str}) para variables categóricas (OHE)...")
        cat_direccionales = []
        for v_cat in vars_cat_ohe:
            idx_var = X_shap.columns.get_loc(v_cat)
            # Evaluar el impacto de la ausencia (0) y la presencia (1) de la condición
            for valor_cat in [0, 1]:
                indices_cat = (X_shap[v_cat] == valor_cat)
                n_pacientes_cat = indices_cat.sum()
                
                if n_pacientes_cat > 0:
                    promedio_crudo = matriz_clase_alta[indices_cat, idx_var].mean()
                    if promedio_crudo > 0: efecto = "Aumenta Mortalidad (+)"
                    elif promedio_crudo < 0: efecto = "Protector / Supervivencia (-)"
                    else: efecto = "Neutral"

                    # Guardar el hallazgo clínico categórico
                    cat_direccionales.append({
                        "Variable": v_cat,
                        "Condicion_OHE": valor_cat,
                        "Significado": "Presencia (1)" if valor_cat == 1 else "Ausencia (0)",
                        "N_Pacientes": n_pacientes_cat,
                        f"SHAP_Crudo_{target_name}_Clase{idx_clase_alta}": promedio_crudo,
                        "Efecto_Clinico": efecto
                    })
                    
        # Exportar el reporte direccional categórico
        df_cat_direccional = pd.DataFrame(cat_direccionales)
        df_cat_direccional.to_csv(os.path.join(dir_sub_enfoque, f"Rangos_Direccion_Categoricas_{target_name}_{sufijo_archivo}.csv"), index=False)

        # 7. Paneles de Dependencia (Top 20)
        # Generar gráficos que cruzan el valor real de la variable vs su impacto SHAP
        print(f"   -> Generando paneles de dependencia en carpeta...")
        top_20_vars = df_shap_imp.head(20).index.tolist()
        for var in top_20_vars:
            if var in X_shap.columns:
                fig, ax = plt.subplots(figsize=(6, 4.5))
                valores_sh_fallecido = matriz_shap[:, :, 1]
                shap.dependence_plot(var, valores_sh_fallecido, X_shap, interaction_index=None, ax=ax, show=False)
                ax.set_title('Impacto en Riesgo de Fallecimiento (Clase 1)', fontsize=10)
                fig.suptitle(f'Dependence Plot: {var} ({sufijo_archivo})', fontsize=11, y=1.02)
                plt.tight_layout()
                plt.savefig(os.path.join(dir_dependence, f"SHAP_Dependence_{var}.png"), dpi=200, bbox_inches='tight')
                plt.close()
                
        # Limpiar la memoria utilizada masivamente en este sub-proceso
        print(f"   Liberando memoria asignada al enfoque {tipo_enfoque}...")
        del X_shap, matriz_shap, df_shap_imp, df_shap_porcentajes
        gc.collect()
        
        # Devolver la tabla de porcentajes formateada para ser usada en comparativas
        return df_retorno

    # -------------------------------------------------------------------------
    # EJECUCIÓN SECUENCIAL Y REPORTE CRUZADO (DIFERENCIAS)
    # -------------------------------------------------------------------------
    
    print("\n--- PASO A: Cargando datos para análisis global ---")
    # Cargar test oncológico y de control para conformar el escenario global de la clínica
    df_onco_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_onco.csv"), low_memory=False)
    df_control_test = pd.read_csv(os.path.join(dir_datos, "dataset_prueba_control.csv"), low_memory=False)
    
    # Lógica de muestreo inteligente: Completar pacientes hasta llegar a 200,000 manteniendo la distribución
    n_onco_total = len(df_onco_test)
    n_control_needed = 200000 - n_onco_total 
    
    # Extraer proporción natural de los controles y muestrear estratificadamente
    proporciones_control = df_control_test[target_name].value_counts(normalize=True)
    df_ctrl_sample = df_control_test.groupby(target_name, group_keys=False).apply(
        lambda x: x.sample(min(len(x), int(np.round(n_control_needed * proporciones_control[x.name]))), random_state=42)
    )
    
    # Ajuste fino: Asegurar matemáticamente que la muestra de control coincida exactamente con la cantidad requerida
    if len(df_ctrl_sample) != n_control_needed:
        if len(df_ctrl_sample) < n_control_needed:
            dif = n_control_needed - len(df_ctrl_sample)
            extras = df_control_test.drop(df_ctrl_sample.index).sample(n=dif, random_state=42)
            df_ctrl_sample = pd.concat([df_ctrl_sample, extras])
        elif len(df_ctrl_sample) > n_control_needed:
            dif = len(df_ctrl_sample) - n_control_needed
            df_ctrl_sample = df_ctrl_sample.drop(df_ctrl_sample.sample(n=dif, random_state=42).index)
            
    # Unificar dataset onco + control y mezclar (shuffle) aleatoriamente
    df_test_global = pd.concat([df_onco_test, df_ctrl_sample], ignore_index=True).sample(frac=1, random_state=42)
    # Liberar memoria de los dataframes intermedios
    del df_control_test, df_ctrl_sample; gc.collect() 
    
    # Disparar la generación de SHAP para el dataset GLOBAL
    df_global_pct = procesar_enfoque_shap_binario(df_test_global, "Global (Onco + Control)", "Valores SHAP (global)")
    
    print("\n--- PASO B: Cargando datos para análisis oncológico ---")
    # Disparar la generación de SHAP para el dataset estrictamente ONCOLÓGICO
    df_onco_pct = procesar_enfoque_shap_binario(df_onco_test, "Oncológico Estricto", "Valores SHAP (oncologicos)")
    
    print("\n--- PASO C: Generando reporte comparativo (Diferencias Top 20 Onco vs Global) ---")
    # Indexar la posición del ranking (Top) de las variables en ambos escenarios
    df_global_pct['Posicion_Global'] = df_global_pct.index + 1
    df_onco_pct['Posicion_Onco'] = df_onco_pct.index + 1
    
    # Aislar las 20 variables más críticas del escenario oncológico para su análisis cruzado
    top_20_onco = df_onco_pct.head(20).copy()
    
    # Cruzar ambos reportes usando el nombre de la variable como llave (Left Join)
    df_comparacion = pd.merge(top_20_onco, df_global_pct, on='Variable', suffixes=('_Onco', '_Global'), how='left')
    # Calcular delta del impacto porcentual unificado
    df_comparacion['Diferencia_Impacto_Total'] = df_comparacion['Impacto_Total_Onco'] - df_comparacion['Impacto_Total_Global']
    
    # Configurar los nombres de las columnas que contienen los impactos por clase
    col_clase = f"Clase_{idx_clase_alta}"
    col_clase_onco = f"{col_clase}_Onco"
    col_clase_global = f"{col_clase}_Global"
    
    # Base de columnas requeridas en el reporte final
    columnas_finales = ['Variable', 'Posicion_Onco', 'Posicion_Global', 'Impacto_Total_Onco', 'Impacto_Total_Global', 'Diferencia_Impacto_Total']
    
    # Calcular y agregar el delta específico del impacto sobre la clase "Fallecido", si existieran en la tabla
    if col_clase_onco in df_comparacion.columns and col_clase_global in df_comparacion.columns:
        nombre_diferencia_clase = f"Diferencia_{col_clase}"
        df_comparacion[nombre_diferencia_clase] = df_comparacion[col_clase_onco] - df_comparacion[col_clase_global]
        columnas_finales.extend([col_clase_onco, col_clase_global, nombre_diferencia_clase])
        
    # Extraer sub-dataset final, formatearlo y exportarlo a disco como CSV
    df_final = df_comparacion[columnas_finales]
    ruta_diferencias = os.path.join(dir_base_resultados, f"Diferencias_SHAP_{target_name}_ONCO_GLOBAL.csv")
    df_final.to_csv(ruta_diferencias, index=False)
    
    # Mensajes de éxito y cierre del script
    print("\n" + "="*80)
    print("PROCESO UNIFICADO FINALIZADO CON ÉXITO")
    print(f"Reportes guardados en: {dir_base_resultados}")
    print("="*80)

In [ ]:
# Ejecuta el pipeline de forma directa
generar_explicabilidad_shap_mortalidad_rf()

INICIANDO FASE 5: SHAP BINARIO - TARGET: MORTALIDAD
Foco clínico de análisis direccional: Mortalidad (Clase 1)
Hora de inicio: 2026-07-16 03:05:58
-> Cargando modelo óptimo pre-entrenado (Modelo_Optimo_RF_MORTALIDAD.pkl)...
-> Inicializando SHAP TreeExplainer nativo...

--- PASO A: Cargando datos para análisis global ---

--- Procesando enfoque: GLOBAL (ONCO + CONTROL) (Datos: 200000) ---
      -> Procesando bloque 1 de 400...
      -> Procesando bloque 10 de 400...
      -> Procesando bloque 20 de 400...
      -> Procesando bloque 30 de 400...
      -> Procesando bloque 40 de 400...
      -> Procesando bloque 50 de 400...
      -> Procesando bloque 60 de 400...
      -> Procesando bloque 70 de 400...
      -> Procesando bloque 80 de 400...
      -> Procesando bloque 90 de 400...
      -> Procesando bloque 100 de 400...
      -> Procesando bloque 110 de 400...
      -> Procesando bloque 120 de 400...
      -> Procesando bloque 130 de 400...
      -> Procesando bloque 140 de 400...
    